# **Conditional VAE — Normal-only Synthetic Data**

One config-driven pipeline for both datasets. Set `DATASET` in the config cell and run all.

1. **Config** — the only dataset-specific values
2. **Column typing + transforms** — inferred from values (fit on normal rows only), mirrors the EDA marginal rules
3. **Model** — conditional VAE with a gated decoder
4. **Training** — normal rows only
5. **Sampling** — categorical combo from the empirical normal joint → decode
6. **Post-hoc filter** — drop candidates that look anomalous / off the normal manifold
   - **6b. Shaping** — per-column sculpting + label-free trim (per-dataset strength, off for NSL)
7. **Write submission** — `id` + original feature columns, exactly N rows
8. **Sanity / explainability checks** — synthetic vs real normal

---
**1. Config**

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from pyod.models.ecod import ECOD

DATASET = "NSL"  # "NSL" | "UNSW"

ROOT = Path("/Users/han./Desktop/CS5344")
CONFIGS = {
    "NSL": dict(path=ROOT / "data/raw/NSL-KDD_train.csv",
                cat_cols=["protocol_type", "service", "flag"], N=40_000,
                # NSL normals are already cleanly separated from attacks: tightening only removes variety.
                # The VAE at temperature 1 is too narrow (std 0.82x real), so rare-but-normal rows raise
                # false alarms; widening fixes that (temperature sweep 1 -> 8: 0.9303 -> 0.9376, plateau)
                clf_threshold=0.5, max_discrete_levels=0, temperature=8.0),
    "UNSW": dict(path=ROOT / "data/raw/UNSW-NB15_train.csv",
                 cat_cols=["proto", "service", "state"], N=70_000,
                 # UNSW normals overlap attacks: prune hard, sculpt, then trim the outskirts
                 clf_threshold=0.01, max_discrete_levels=0, sculpt_keep=0.35, trim_frac=0.75),
}
COMMON = dict(
    label="is_anomaly",
    seed=42,
    # model / training
    latent_dim=16, hidden=256, epochs=300, batch_size=512, lr=1e-3,
    beta_warmup_frac=0.2, val_frac=0.1, patience=30,
    # sampling
    temperature=1.0,        # z ~ N(0, tau^2 I); lower = tighter normal core
    sample_values=True,     # sample from the value Gaussian (False = use its mean)
    # post-hoc filter
    oversample=1.5,         # candidates generated per round = oversample * N
    clf_threshold=0.5,      # keep if P(anomaly) < threshold
    ecod_quantile=0.99,     # keep if ECOD score <= this quantile of real normal scores
    # True = filter classifier sees only anomaly half A, so "pseudo kaggle.ipynb" can score
    # honestly on half B. Use True for local experiments, False for the real submission.
    clf_holdout=False,
    # integer columns with 3..max_discrete_levels distinct values are modelled as categories
    # (softmax over the observed values) instead of a smooth Gaussian. 0 = off.
    max_discrete_levels=16,
    # 6b. shaping (both off by default; per-dataset values in CONFIGS)
    sculpt_keep=1.0,        # keep the lowest `sculpt_keep` fraction by per-column attack score (1.0 = off)
    trim_frac=0.0,          # then drop the `trim_frac` most outlying rows by ECOD on real normals (0.0 = off)
)
cfg = {**COMMON, **CONFIGS[DATASET]}
tag = (f"clf{cfg['clf_threshold']}" + (f"_d{cfg['max_discrete_levels']}" if cfg["max_discrete_levels"] else "")
       + (f"_t{cfg['temperature']}" if cfg["temperature"] != 1.0 else "")
       + (f"_k{cfg['sculpt_keep']}" if cfg["sculpt_keep"] < 1.0 else "")
       + (f"_tr{cfg['trim_frac']}" if cfg["trim_frac"] > 0 else "")
       + ("_holdout" if cfg["clf_holdout"] else ""))
cfg["out_path"] = ROOT / f"data/processed/synthetic_{DATASET}_{tag}.csv"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(cfg["seed"])
rng = np.random.default_rng(cfg["seed"])
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

df = pd.read_csv(cfg["path"])
cat_cols = cfg["cat_cols"]
feature_cols = [c for c in df.columns if c != cfg["label"]]   # original order, used for output
num_cols = [c for c in feature_cols if c not in cat_cols]
normal_df = df[df[cfg["label"]] == 0].reset_index(drop=True)
print(DATASET, df.shape, "normal rows:", len(normal_df), "N:", cfg["N"], "device:", device)

---
**2. Column typing + transforms**

Each numeric column is typed from its values on normal rows (never by name), so the same code works for both datasets:

| type | rule | gate | value modelled |
|---|---|---|---|
| `const` | one unique value | — | dropped, re-inserted at the end |
| `binary` | values ⊆ {0,1} | 2-way (0 / 1) | — |
| `discrete` | integers with ≤ `max_discrete_levels` distinct values | K-way softmax over the observed values | — (only real values can be generated) |
| `rate` | in [0,1], not only 0/1 | 3-way (0 / 1 / mid) | `logit(x)` for mid, standardised |
| `zinf` | ≥ 0 otherwise | 2-way (0 / nonzero) | `log1p(x)` for nonzero, standardised |
| `cont` | has negatives | — | standardised `x` |

We also record each column's decimal precision (integers = 0) and min/max for rounding/clipping at sampling time.

In [ ]:
EPS = 1e-4

def decimals_of(x, max_dec=6):
    for d in range(max_dec + 1):
        if np.allclose(np.round(x, d), x, rtol=0, atol=1e-9):
            return d
    return None  # genuinely continuous: no rounding

def fit_column_specs(nd):
    specs = []
    for c in num_cols:
        x = nd[c].to_numpy(dtype=float)
        u = np.unique(x)
        s = dict(name=c, min=x.min(), max=x.max(), dec=decimals_of(x))
        if len(u) == 1:
            s.update(kind="const", value=u[0])
        elif set(u) <= {0.0, 1.0}:
            s.update(kind="binary", K=2)
        elif s["dec"] == 0 and len(u) <= cfg["max_discrete_levels"]:
            s.update(kind="discrete", K=len(u), levels=u)
        elif x.min() >= 0 and x.max() <= 1:
            mid = x[(x > 0) & (x < 1)]
            t = np.log(np.clip(mid, EPS, 1 - EPS) / (1 - np.clip(mid, EPS, 1 - EPS)))
            s.update(kind="rate", K=3, mu=t.mean() if len(t) else 0.0, sd=t.std() if len(t) > 1 else 1.0)
        elif x.min() >= 0:
            t = np.log1p(x[x > 0])
            s.update(kind="zinf", K=2, mu=t.mean(), sd=t.std() if len(t) > 1 else 1.0)
        else:
            s.update(kind="cont", mu=x.mean(), sd=x.std())
        if "sd" in s and s["sd"] < 1e-6:
            s["sd"] = 1.0
        specs.append(s)
    return specs

specs = fit_column_specs(normal_df)
const_specs = [s for s in specs if s["kind"] == "const"]
model_specs = [s for s in specs if s["kind"] != "const"]
gate_specs  = [s for s in model_specs if "K" in s]                     # columns with a gate head
value_specs = [s for s in model_specs if s["kind"] in ("rate", "zinf", "cont")]  # columns with a value head

print(pd.Series([s["kind"] for s in specs]).value_counts().to_string())
print("const:", [s["name"] for s in const_specs])
print("rate :", [s["name"] for s in specs if s["kind"] == "rate"])
print("discrete:", [(s["name"], s["K"]) for s in specs if s["kind"] == "discrete"])

def encode_numeric(nd):
    """Returns gate targets (n, n_gate) int, value targets (n, n_val), value mask (n, n_val)."""
    n = len(nd)
    G = np.zeros((n, len(gate_specs)), dtype=np.int64)
    for j, s in enumerate(gate_specs):
        x = nd[s["name"]].to_numpy(dtype=float)
        if s["kind"] == "rate":
            G[:, j] = np.where(x == 0, 0, np.where(x == 1, 1, 2))
        elif s["kind"] == "discrete":
            G[:, j] = np.searchsorted(s["levels"], x)
        else:  # binary / zinf: 0 = zero, 1 = one / nonzero
            G[:, j] = (x != 0).astype(np.int64)
    V = np.zeros((n, len(value_specs)), dtype=np.float64)
    M = np.zeros((n, len(value_specs)), dtype=np.float32)
    for j, s in enumerate(value_specs):
        x = nd[s["name"]].to_numpy(dtype=float)
        if s["kind"] == "rate":
            m = (x > 0) & (x < 1)
            xc = np.clip(x, EPS, 1 - EPS)
            t = np.log(xc / (1 - xc))
        elif s["kind"] == "zinf":
            m = x > 0
            t = np.log1p(np.maximum(x, 0))
        else:
            m = np.ones(n, dtype=bool)
            t = x
        V[:, j] = np.where(m, (t - s["mu"]) / s["sd"], 0.0)
        M[:, j] = m
    return G, V, M

def decode_numeric(G, V):
    """Inverse of encode_numeric -> DataFrame of numeric cols (incl. const), rounded + clipped."""
    n = G.shape[0]
    out = {}
    gi = {s["name"]: j for j, s in enumerate(gate_specs)}
    vi = {s["name"]: j for j, s in enumerate(value_specs)}
    for s in specs:
        k, c = s["kind"], s["name"]
        if k == "const":
            x = np.full(n, s["value"])
        elif k == "binary":
            x = G[:, gi[c]].astype(float)
        elif k == "discrete":
            x = s["levels"][G[:, gi[c]]]
        elif k == "rate":
            mid = 1 / (1 + np.exp(-(V[:, vi[c]] * s["sd"] + s["mu"])))
            g = G[:, gi[c]]
            x = np.where(g == 0, 0.0, np.where(g == 1, 1.0, mid))
        elif k == "zinf":
            nz = np.expm1(V[:, vi[c]] * s["sd"] + s["mu"])
            x = np.where(G[:, gi[c]] == 0, 0.0, nz)
        else:
            x = V[:, vi[c]] * s["sd"] + s["mu"]
        x = np.clip(x, s["min"], s["max"])
        if s["dec"] is not None:
            x = np.round(x, s["dec"])
        if k == "zinf" and s["dec"] is not None:
            # a "nonzero" draw must not round down to 0 (keeps the gate meaningful)
            step = 10.0 ** -s["dec"]
            x = np.where((G[:, gi[c]] == 1) & (x == 0), step, x)
        out[c] = x
    return pd.DataFrame(out)

# conditions: per-column one-hot of the categoricals (categories seen in NORMAL rows only)
cat_levels = {c: sorted(normal_df[c].unique()) for c in cat_cols}
def encode_cats(cd):
    parts = []
    for c in cat_cols:
        codes = pd.Categorical(cd[c], categories=cat_levels[c]).codes
        oh = np.zeros((len(cd), len(cat_levels[c])), dtype=np.float32)
        oh[np.arange(len(cd)), codes] = 1.0
        parts.append(oh)
    return np.concatenate(parts, axis=1)

G_all, V_all, M_all = encode_numeric(normal_df)
C_all = encode_cats(normal_df)
print("gate cols:", G_all.shape[1], "value cols:", V_all.shape[1], "cond dim:", C_all.shape[1])

# round-trip check: encode -> decode must reproduce the real normal rows
rt = decode_numeric(G_all, V_all)
max_err = (rt[num_cols] - normal_df[num_cols]).abs().max()
print("round-trip max abs error (worst 3):")
print(max_err.sort_values(ascending=False).head(3).to_string())

---
**3. Model — Conditional VAE with gated decoder**

- **Encoder** `q(z | x, c)`: input = one-hot gates + transformed values (zeroed where the gate says "no value") + condition `c`.
- **Decoder** `p(x | z, c)`: per column a gate head (2-way, 3-way or K-way softmax for `discrete` columns) and, for `rate`/`zinf`/`cont`, a value mean with a learned per-column σ.
- The decoder's factorisation per column is exactly the EDA marginal rule (zero/one spikes + a smooth part on `log1p`/`logit` scale); **dependence between columns comes only through the shared `z` and `c`**.

In [ ]:
gate_K = [s["K"] for s in gate_specs]
# gate columns grouped by number of classes, so each group is one batched softmax
K_groups = {k: [j for j, kk in enumerate(gate_K) if kk == k] for k in sorted(set(gate_K))}
in_dim = sum(gate_K) + len(value_specs)
cond_dim = C_all.shape[1]

def mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h), nn.LeakyReLU(0.2), nn.Linear(h, h), nn.LeakyReLU(0.2), nn.Linear(h, o))

class CVAE(nn.Module):
    def __init__(self, latent, hidden):
        super().__init__()
        self.latent = latent
        self.enc = mlp(in_dim + cond_dim, hidden, 2 * latent)
        self.dec = mlp(latent + cond_dim, hidden, sum(gate_K) + len(value_specs))
        self.log_sigma = nn.Parameter(torch.zeros(len(value_specs)))

    def encoder_input(self, G, V, M):
        oh = [F.one_hot(G[:, j], k).float() for j, k in enumerate(gate_K)]
        return torch.cat(oh + [V * M], dim=1)

    def encode(self, x, c):
        mu, logvar = self.enc(torch.cat([x, c], 1)).chunk(2, dim=1)
        return mu, logvar.clamp(-10, 10)

    def decode(self, z, c):
        h = self.dec(torch.cat([z, c], 1))
        n, off, logits = h.shape[0], 0, {}
        for k, idx in K_groups.items():
            logits[k] = h[:, off:off + k * len(idx)].reshape(n, len(idx), k)
            off += k * len(idx)
        return logits, h[:, off:]

    def forward(self, G, V, M, c):
        mu, logvar = self.encode(self.encoder_input(G, V, M), c)
        z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()
        return self.decode(z, c), mu, logvar

def loss_fn(model, out, mu, logvar, G, V, M, beta):
    logits, v_mu = out
    rec = 0.0
    for k, idx in K_groups.items():
        rec = rec + F.cross_entropy(logits[k].reshape(-1, k), G[:, idx].reshape(-1), reduction="sum")
    ls = model.log_sigma
    nll = 0.5 * ((V - v_mu) / ls.exp()) ** 2 + ls + 0.5 * np.log(2 * np.pi)
    rec = rec + (nll * M).sum()
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum()
    n = G.shape[0]
    return (rec + beta * kl) / n, rec / n, kl / n

model = CVAE(cfg["latent_dim"], cfg["hidden"]).to(device)
print(model)
print("params:", sum(p.numel() for p in model.parameters()))

---
**4. Training (normal rows only)**

β warms up 0→1 over the first 20% of epochs (avoids posterior collapse early). Early stopping on a 10% split of normal rows, using the full-β loss.

In [ ]:
T = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt, device=device)
perm = rng.permutation(len(normal_df))
n_val = int(cfg["val_frac"] * len(perm))
tr_idx, va_idx = perm[n_val:], perm[:n_val]
G_t, V_t, M_t, C_t = T(G_all, torch.long), T(V_all), T(M_all), T(C_all)

opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
warm = max(1, int(cfg["beta_warmup_frac"] * cfg["epochs"]))
hist = {"train": [], "val": [], "rec": [], "kl": []}
best, best_state, bad = np.inf, None, 0

for epoch in range(cfg["epochs"]):
    beta = min(1.0, epoch / warm)
    model.train()
    order = tr_idx[rng.permutation(len(tr_idx))]
    tot = 0.0
    for i in range(0, len(order), cfg["batch_size"]):
        b = torch.as_tensor(order[i:i + cfg["batch_size"]], device=device)
        out, mu, logvar = model(G_t[b], V_t[b], M_t[b], C_t[b])
        loss, _, _ = loss_fn(model, out, mu, logvar, G_t[b], V_t[b], M_t[b], beta)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    model.eval()
    with torch.no_grad():
        b = torch.as_tensor(va_idx, device=device)
        out, mu, logvar = model(G_t[b], V_t[b], M_t[b], C_t[b])
        vloss, vrec, vkl = loss_fn(model, out, mu, logvar, G_t[b], V_t[b], M_t[b], 1.0)
    hist["train"].append(tot / len(tr_idx)); hist["val"].append(vloss.item())
    hist["rec"].append(vrec.item()); hist["kl"].append(vkl.item())
    if epoch >= warm:  # only compare full-beta losses
        if vloss.item() < best - 1e-4:
            best, bad = vloss.item(), 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"early stop at epoch {epoch}")
                break
    if epoch % 25 == 0:
        print(f"ep {epoch:3d} beta={beta:.2f} train={hist['train'][-1]:.3f} val={vloss.item():.3f} rec={vrec.item():.3f} kl={vkl.item():.3f}")

if best_state is not None:
    model.load_state_dict(best_state)
print("best val loss:", best)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["train"], label="train"); axes[0].plot(hist["val"], label="val (beta=1)")
axes[0].axvline(warm, ls="--", c="grey"); axes[0].set_title("loss"); axes[0].legend()
axes[1].plot(hist["rec"], label="val recon"); axes[1].plot(hist["kl"], label="val KL")
axes[1].set_title("val components"); axes[1].legend()
plt.tight_layout(); plt.show()

---
**5. Sampling**

1. Draw the categorical combo from the **empirical joint** of normal rows, so combos stay realistic and rare combos keep their true frequency.
2. Draw `z ~ N(0, τ²I)` from the prior.
3. Decode, sample each gate from its softmax, then sample the value from its Gaussian.
4. Invert the transforms, clip to the normal-row range, round to the column's precision, and re-insert the constant columns.

In [ ]:
combo_df = normal_df[cat_cols]

@torch.no_grad()
def generate(n):
    cats = combo_df.iloc[rng.integers(0, len(combo_df), n)].reset_index(drop=True)
    c = T(encode_cats(cats))
    z = torch.randn(n, cfg["latent_dim"], device=device) * cfg["temperature"]
    model.eval()
    logits, v_mu = model.decode(z, c)
    G = np.zeros((n, len(gate_specs)), dtype=np.int64)
    for k, idx in K_groups.items():
        G[:, idx] = torch.distributions.Categorical(logits=logits[k]).sample().cpu().numpy()
    V = v_mu
    if cfg["sample_values"]:
        V = V + torch.randn_like(V) * model.log_sigma.exp()
    nums = decode_numeric(G, V.cpu().numpy().astype(np.float64))
    return pd.concat([cats, nums], axis=1)[feature_cols]

demo = generate(5)
demo

---
**6. Post-hoc filter (the only place anomaly rows are used)**

Generate about 1.5×N candidates per round and keep a candidate only if **both** checks pass:
- **(a) Classifier:** `HistGradientBoostingClassifier` trained on train normal vs anomaly gives P(anomaly) < threshold. This removes synthetic "normals" that drifted toward known attack regions.
- **(b) Manifold:** ECOD fitted on real normal rows (one-hot + StandardScaler) gives a score ≤ the 99th percentile of the real normal rows' own scores. This removes samples off the normal manifold.

Repeat rounds until enough rows survive: N, or more when the shaping step (6b) will drop some, so the final N rows are all fresh (no repeats).

With `clf_holdout=True` the classifier only sees anomaly half A, so `pseudo kaggle.ipynb` can score the output on the unseen half B without leakage.

In [ ]:
# (a) normal-vs-anomaly classifier, native categorical support via ordinal codes
all_levels = {c: sorted(df[c].unique()) for c in cat_cols}
def clf_matrix(d):
    X = d[feature_cols].copy()
    for c in cat_cols:
        X[c] = pd.Categorical(X[c], categories=all_levels[c])
    return X

clf = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=cfg["seed"])
if cfg["clf_holdout"]:
    # same split as "pseudo kaggle.ipynb": shuffle anomalies with random_state=0, half A = first half
    anom = df[df[cfg["label"]] == 1].sample(frac=1, random_state=0)
    clf_train = pd.concat([normal_df, anom.iloc[: len(anom) // 2]], ignore_index=True)
else:
    clf_train = df
clf.fit(clf_matrix(clf_train), clf_train[cfg["label"]])

# (b) ECOD on real normals
def onehot_matrix(d):
    X = pd.get_dummies(d[feature_cols], columns=cat_cols, dtype=float)
    return X.reindex(columns=ohe_cols, fill_value=0.0).to_numpy()

ohe_cols = pd.get_dummies(normal_df[feature_cols], columns=cat_cols, dtype=float).columns
scaler = StandardScaler().fit(onehot_matrix(normal_df))
ecod = ECOD().fit(scaler.transform(onehot_matrix(normal_df)))
ecod_cut = np.quantile(ecod.decision_scores_, cfg["ecod_quantile"])

# 6b drops rows later, so collect enough survivors that exactly N fresh rows remain
target = int(np.ceil(cfg["N"] / (cfg["sculpt_keep"] * (1 - cfg["trim_frac"]))))
kept, stats = [], []
n_kept, rounds = 0, 0
while n_kept < target:
    cand = generate(int(cfg["oversample"] * cfg["N"]))
    p_anom = clf.predict_proba(clf_matrix(cand))[:, 1]
    e = ecod.decision_function(scaler.transform(onehot_matrix(cand)))
    ok_a, ok_b = p_anom < cfg["clf_threshold"], e <= ecod_cut
    keep = ok_a & ok_b
    stats.append(dict(round=rounds, candidates=len(cand), rej_clf=(~ok_a).mean(),
                      rej_ecod=(~ok_b).mean(), kept=keep.mean()))
    kept.append(cand[keep]); n_kept += keep.sum(); rounds += 1
    if rounds >= 50:
        raise RuntimeError("filter keeps too little — relax thresholds or check the model")

print(pd.DataFrame(stats).round(4).to_string(index=False))
pool = pd.concat(kept, ignore_index=True)
print("filtered pool:", len(pool), "target:", target)

---
**6b. Shaping: per-column sculpting + label-free trim**

The guards (ECOD, IForest) judge one column at a time, so the pool is shaped column by column. Both steps are optional, and their strength is a per-dataset config.

- **Sculpt (uses attack rows):** cut each column into 20 quantile bins (raw values or categories if the column has few). Per bin, LLR = log P(bin | attack) − log P(bin | normal), Laplace-smoothed. Row score = Σ over columns of max(0, LLR), i.e. how many "attack-flavoured" column values the row has. Keep the lowest `sculpt_keep` fraction. This empties the bins where attacks live, so attacks with an attack-flavoured value in *any* column land in sparse territory.
- **Trim (label-free):** score the sculpted rows with the ECOD fitted on real normals (same as filter (b)) and drop the `trim_frac` most outlying. This tightens normal around its core, which helps against *unseen* attack types where normals and attacks overlap (UNSW), but only removes real variety where they are already separated (NSL, so off there).

Then take exactly N rows at random. Every row is a distinct VAE draw, never a repeat.

In [ ]:
def sculpt_score(pool, normals, attacks, n_bins=20):
    """Sum over columns of max(0, log P(bin|attack) - log P(bin|normal))."""
    score = np.zeros(len(pool))
    for c in feature_cols:
        if c in cat_cols:
            binned = lambda d: d[c].astype(str).to_numpy()
        else:
            v = pd.concat([normals[c], attacks[c]]).to_numpy()
            edges = (None if len(np.unique(v)) <= n_bins
                     else np.unique(np.quantile(v, np.linspace(0, 1, n_bins + 1)[1:-1])))
            binned = (lambda d: d[c].to_numpy()) if edges is None else \
                     (lambda d, e=edges: np.searchsorted(e, d[c].to_numpy(), side="right"))
        bn, ba = pd.Series(binned(normals)).value_counts(), pd.Series(binned(attacks)).value_counts()
        keys = bn.index.union(ba.index); k = len(keys)
        pn = (bn.reindex(keys, fill_value=0) + 1) / (len(normals) + k)
        pa = (ba.reindex(keys, fill_value=0) + 1) / (len(attacks) + k)
        score += np.maximum(pd.Series(binned(pool)).map(np.log(pa / pn)).fillna(0).to_numpy(), 0)
    return score

def keep_lowest(d, score, frac):
    n = int(np.ceil(frac * len(d)))
    return d.iloc[np.sort(np.argsort(score, kind="stable")[:n])].reset_index(drop=True)

attacks = clf_train[clf_train[cfg["label"]] == 1]   # the same rows the filter classifier saw
if cfg["sculpt_keep"] < 1.0:
    pool = keep_lowest(pool, sculpt_score(pool, normal_df, attacks), cfg["sculpt_keep"])
    print("after sculpt:", len(pool))
if cfg["trim_frac"] > 0:
    pool = keep_lowest(pool, ecod.decision_function(scaler.transform(onehot_matrix(pool))), 1 - cfg["trim_frac"])
    print("after trim:", len(pool))

assert len(pool) >= cfg["N"]
synth = pool.iloc[rng.permutation(len(pool))[:cfg["N"]]].reset_index(drop=True)
print("final synthetic:", synth.shape)

---
**7. Write submission**: `id`, then exactly the original feature columns, with no label column.

In [ ]:
sub = synth[feature_cols].copy()
for c in num_cols:  # keep integer columns as ints in the CSV
    if pd.api.types.is_integer_dtype(df[c]):
        sub[c] = sub[c].round().astype("int64")
sub.insert(0, "id", np.arange(len(sub)))

assert len(sub) == cfg["N"]
assert list(sub.columns) == ["id"] + feature_cols
assert cfg["label"] not in sub.columns
assert not sub.isna().any().any()
assert sub["id"].is_unique
for c in cat_cols:
    assert set(sub[c]) <= set(normal_df[c]), c

cfg["out_path"].parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(cfg["out_path"], index=False)
print("wrote", cfg["out_path"], sub.shape)

---
**8. Sanity and explainability checks (synthetic vs real normal rows)**

These check the distributional properties of the generated data. They are not a held-out evaluation.

In [ ]:
real, fake = normal_df[feature_cols], sub[feature_cols]

# 8a. zero-rate per numeric column
zr = pd.DataFrame({"real": real[num_cols].eq(0).mean(), "synth": fake[num_cols].eq(0).mean()})
zr["abs_diff"] = (zr.real - zr.synth).abs()
print("zero-rate: worst 10 columns")
print(zr.sort_values("abs_diff", ascending=False).head(10).round(4).to_string())

# 8b. quantiles per numeric column
qs = [0.5, 0.9, 0.99]
qt = pd.concat({"real": real[num_cols].quantile(qs).T, "synth": fake[num_cols].quantile(qs).T}, axis=1)
print("\nquantiles (first 15 columns)")
print(qt.head(15).round(3).to_string())

In [ ]:
# 8c. correlation structure
cr, cs = real[num_cols].corr(), fake[num_cols].corr()
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, m, t in zip(axes, [cr, cs, (cs - cr)], ["real normal", "synthetic", "synthetic - real"]):
    sns.heatmap(m, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, xticklabels=False, yticklabels=False)
    ax.set_title(t)
plt.tight_layout(); plt.show()
print("mean |corr diff|:", np.nanmean(np.abs((cs - cr).to_numpy())).round(4))

# strongest real pairs, and whether the synthetic data preserves them
pairs = cr.where(np.triu(np.ones(cr.shape, bool), 1)).stack()
top = pairs.abs().sort_values(ascending=False).head(12).index
print(pd.DataFrame({"real": pairs[top], "synth": [cs.loc[a, b] for a, b in top]}).round(3).to_string())

In [ ]:
# 8d. categorical combo frequencies
cf = pd.DataFrame({"real": real.groupby(cat_cols).size() / len(real),
                   "synth": fake.groupby(cat_cols).size() / len(fake)}).fillna(0)
print("combo frequency (top 10 by real):")
print(cf.sort_values("real", ascending=False).head(10).round(4).to_string())
print("total variation distance:", (0.5 * (cf.real - cf.synth).abs().sum()).round(4))